# Project Short — Notebook Kaggle propre

Ce notebook regroupe le workflow utilisé pour générer une courte vidéo à partir de l'image de Lina avec LTX-Video.

## Utilisation

Lors d'une nouvelle session Kaggle :
1. Exécuter les cellules 1 à 4 une seule fois.
2. Vérifier/modifier les paramètres dans la cellule 5.
3. Exécuter la cellule 6 pour générer la vidéo.
4. Pour une nouvelle génération dans la même session, modifier uniquement la cellule 5 puis relancer la cellule 6.

Le chemin actuel de l'image de Lina est déjà configuré :
`/kaggle/input/datasets/pathemandiouba/lina-t/Petite fille dans un village mditerranen.png`


In [1]:
# CELLULE 1 — Vérifier le GPU Kaggle

import torch
import subprocess

subprocess.run(["nvidia-smi"])

print("\nPyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i} :", torch.cuda.get_device_name(i))
    print("VRAM :", round(props.total_memory / 1024**3, 2), "GB")


Wed Oct  7 08:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# CELLULE 2 — Installer les dépendances
# À exécuter une seule fois par nouvelle session Kaggle.

import subprocess
import sys

packages = [
    "diffusers",
    "transformers",
    "accelerate",
    "sentencepiece",
    "ftfy",
    "imageio",
    "imageio-ffmpeg",
    "huggingface_hub",
    "av",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", *packages],
    check=True
)

print("✅ Dépendances installées.")


  Using cached diffusers-0.41.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.19.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached diffusers-0.41.0-py3-none-any.whl (6.1 MB)
Using cached transformers-5.19.0-py3-none-any.whl (12.6 MB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 48.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.39.0
    Uninstalling diff

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ltx-video 0.1.2 requires huggingface-hub~=0.30, but you have huggingface-hub 1.33.0 which is incompatible.
ltx-video 0.1.2 requires transformers<4.52.0,>=4.47.2, but you have transformers 5.19.0 which is incompatible.


In [3]:
# CELLULE 3 — Télécharger / préparer LTX-Video
# À exécuter une seule fois par nouvelle session Kaggle.

import os
import subprocess
import sys

repo_dir = "/kaggle/working/LTX-Video"

if not os.path.isdir(repo_dir):
    subprocess.run(
        ["git", "clone", "https://github.com/Lightricks/LTX-Video.git", repo_dir],
        check=True
    )
else:
    print("Le dépôt LTX-Video existe déjà :", repo_dir)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo_dir}[inference-script]"],
    check=True
)

print("✅ LTX-Video prêt.")


Le dépôt LTX-Video existe déjà : /kaggle/working/LTX-Video
Obtaining file:///kaggle/working/LTX-Video
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of diffusers to determine which version is compatible with other requirements. This could take a while.
  Using cached diffusers-0.40.0-py3-none-any.whl.metadata (20 kB)
  Using cached diffusers-0.39.0-py3-none-any.whl.

  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached diffusers-0.39.0-py3-none-any.whl (5.6 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Building editable for ltx-video (pyproject.toml): started
  Building editable for ltx-video (pyproject.toml): finished with status 'done'
  Created wheel for ltx-video: filename=ltx_video-0.1.2-0.editable-py3-none-any.whl size=15397 sha256=503a67c771d531ed4eaba2c30763fa8dcb3dd3d80a674aeea6f3cc73333a4120
  Stored in directory: /tmp/pip-ephem-wheel-cache-xuul0gqm/wheels/ff/80/94/9dff4b4647460f1854145f8dec3c00887f0b8461c9484c3769
Successfully built ltx-video
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
# CELLULE 4 — Télécharger les fichiers nécessaires
# À exécuter une seule fois par nouvelle session Kaggle.

import os
from huggingface_hub import hf_hub_download

repo_dir = "/kaggle/working/LTX-Video"
checkpoint_dir = os.path.join(repo_dir, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint = hf_hub_download(
    repo_id="Lightricks/LTX-Video",
    filename="ltxv-2b-0.9.8-distilled.safetensors",
    local_dir=checkpoint_dir
)

upscaler = hf_hub_download(
    repo_id="Lightricks/LTX-Video",
    filename="ltxv-spatial-upscaler-0.9.8.safetensors",
    local_dir=checkpoint_dir
)

print("✅ Checkpoint :", checkpoint)
print("✅ Upscaler   :", upscaler)


✅ Checkpoint : /kaggle/working/LTX-Video/checkpoints/ltxv-2b-0.9.8-distilled.safetensors
✅ Upscaler   : /kaggle/working/LTX-Video/checkpoints/ltxv-spatial-upscaler-0.9.8.safetensors


In [8]:
import glob

images = []

for ext in ["png", "jpg", "jpeg", "webp"]:
    images.extend(
        glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)
    )

print("Images trouvées :")

for i, path in enumerate(images):
    print(i, ":", path)

IMAGE_PATH = images[0]
print("\n✅ Image utilisée :", IMAGE_PATH)

Images trouvées :
0 : /kaggle/input/datasets/pathemandiouba/lina-testt/Petite fille dans un village mditerranen.png

✅ Image utilisée : /kaggle/input/datasets/pathemandiouba/lina-testt/Petite fille dans un village mditerranen.png


In [9]:
# CELLULE 5 — PARAMÈTRES DE LA GÉNÉRATION
# Pour chaque nouvelle vidéo, c'est principalement cette cellule qu'il faut modifier.

IMAGE_PATH = "/kaggle/input/datasets/pathemandiouba/lina-t/Petite fille dans un village mditerranen.png"

PROMPT = '''
Lina stands naturally and looks toward the camera.
She blinks softly, smiles and gently waves with one hand.
Subtle breathing and very slight natural hair movement.

Preserve exactly the same character from the input image:
same face, same blonde hair, same pink shirt with heart,
same light blue pants with flowers, same pink shoes,
same proportions and same colors.

The background must remain completely stable and sharp.
Flowers and plants must remain completely still.
No wind.
No moving flowers.
No background motion.
No camera movement.
No morphing.
No character redesign.

Smooth fluid premium 2.5D children's animation.
'''.strip()

NEGATIVE_PROMPT = '''
camera movement, zoom, pan, tilt, camera shake,
moving background, moving flowers, moving plants, wind,
flickering, morphing, face deformation, body deformation,
extra limbs, inconsistent character, blurry, jittery, distorted,
changing clothes, changing hairstyle
'''.strip()

# Format vertical
HEIGHT = 768
WIDTH = 448

# 49 frames à 16 fps ≈ 3,06 secondes
NUM_FRAMES = 49
FRAME_RATE = 16

SEED = 42

# Dossier de sortie. Change le nom pour chaque nouveau plan.
OUTPUT_PATH = "/kaggle/working/lina_test"

print("✅ Paramètres prêts.")
print("Image :", IMAGE_PATH)
print("Sortie :", OUTPUT_PATH)
print("Durée approximative :", round(NUM_FRAMES / FRAME_RATE, 2), "secondes")


✅ Paramètres prêts.
Image : /kaggle/input/datasets/pathemandiouba/lina-t/Petite fille dans un village mditerranen.png
Sortie : /kaggle/working/lina_test
Durée approximative : 3.06 secondes


In [6]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("✅ Cache GPU nettoyé.")
print(
    "VRAM utilisée :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

✅ Cache GPU nettoyé.
VRAM utilisée : 0.0 GB


In [7]:
from pathlib import Path

file_path = Path("/kaggle/working/LTX-Video/ltx_video/inference.py")

text = file_path.read_text()

# 1. Garder le text encoder sur CPU
text = text.replace(
    "text_encoder = text_encoder.to(device)",
    'text_encoder = text_encoder.to("cpu")'
)

# 2. Empêcher LTX de remettre tout le pipeline sur le GPU
text = text.replace(
    "pipeline = pipeline.to(device)\nreturn pipeline",
    """# Do not move the complete pipeline to GPU on low-VRAM cards.
# Components will be handled by the pipeline/offload logic.
return pipeline"""
)

file_path.write_text(text)

print("✅ Patch mémoire appliqué à LTX.")

✅ Patch mémoire appliqué à LTX.


In [10]:
# CELLULE 6 — GÉNÉRER ET AFFICHER LA VIDÉO
# Tant que la session Kaggle reste ouverte, pour une nouvelle génération :
# modifie la cellule 5 puis relance uniquement cette cellule.

import os
import glob
import subprocess
from IPython.display import Video, display

repo_dir = "/kaggle/working/LTX-Video"

if not os.path.isfile(IMAGE_PATH):
    raise FileNotFoundError(
        f"Image introuvable : {IMAGE_PATH}\n"
        "Vérifie le chemin du fichier dans Kaggle."
    )

os.makedirs(OUTPUT_PATH, exist_ok=True)

cmd = [
    "python",
    "inference.py",
    "--prompt", PROMPT,
    "--negative_prompt", NEGATIVE_PROMPT,
    "--conditioning_media_paths", IMAGE_PATH,
    "--conditioning_start_frames", "0",
    "--height", str(HEIGHT),
    "--width", str(WIDTH),
    "--num_frames", str(NUM_FRAMES),
    "--frame_rate", str(FRAME_RATE),
    "--seed", str(SEED),
    "--offload_to_cpu",
    "--output_path", OUTPUT_PATH,
    "--pipeline_config", "configs/ltxv-2b-0.9.8-distilled.yaml",
]

print("🎬 Génération de la vidéo...")
print("Commande préparée avec :")
print(" - image :", IMAGE_PATH)
print(" - résolution :", f"{WIDTH}x{HEIGHT}")
print(" - frames :", NUM_FRAMES)
print(" - fps :", FRAME_RATE)
print(" - sortie :", OUTPUT_PATH)

subprocess.run(cmd, cwd=repo_dir, check=True)

# Recherche du MP4 généré
videos = glob.glob(os.path.join(OUTPUT_PATH, "**", "*.mp4"), recursive=True)

if not videos:
    # Certains scripts peuvent écrire un MP4 voisin du chemin de sortie.
    videos = glob.glob(OUTPUT_PATH + "*.mp4")

if not videos:
    print("⚠️ La commande s'est terminée, mais aucun MP4 n'a été trouvé automatiquement.")
    print("Regarde le panneau Files de Kaggle dans :", OUTPUT_PATH)
else:
    videos.sort(key=os.path.getmtime, reverse=True)
    video_path = videos[0]
    print("✅ Vidéo générée :", video_path)
    display(Video(video_path, embed=True))


FileNotFoundError: Image introuvable : /kaggle/input/datasets/pathemandiouba/lina-t/Petite fille dans un village mditerranen.png
Vérifie le chemin du fichier dans Kaggle.

## Générer une nouvelle vidéo

Dans la **même session Kaggle**, il n'est pas nécessaire de tout recommencer.

Modifie seulement la cellule 5 :
- `IMAGE_PATH` si tu changes de personnage ou de référence ;
- `PROMPT` pour changer le mouvement ;
- `NEGATIVE_PROMPT` si nécessaire ;
- `OUTPUT_PATH` pour donner un nouveau nom au plan ;
- `NUM_FRAMES` / `FRAME_RATE` si tu veux modifier la durée.

Puis relance uniquement la cellule 6.

### Exemple de deuxième mouvement

Tu peux remplacer `PROMPT` par :

```text
Lina hears a soft sound coming from the window.
Her eyes move toward the window first, then she gently turns her head.
She looks curious while keeping the same face, hair, clothes and proportions.
The camera remains locked. The background, flowers and plants remain completely still.
No wind. Smooth natural character animation.
```

Et mettre :

```python
OUTPUT_PATH = "/kaggle/working/lina_turn_window"
```

Si Kaggle redémarre complètement la session, il faudra relancer les cellules 1 à 4 avant de générer à nouveau.
